# 06 — Generate Report
Render `reports/report.md` from evaluation results and convert to PDF.

In [ ]:
import sys
sys.path.insert(0, '..')

import json
from pathlib import Path
from jinja2 import Environment, FileSystemLoader

from src.config import EVAL_RESULTS_JSON, REPORTS_DIR, ARTIFACTS_DIR

## Load evaluation results

In [ ]:
with open(EVAL_RESULTS_JSON) as f:
    results = json.load(f)

ae = results['autoencoder']
cls = results['classifier']

print('Loaded eval results.')
print(f"  AE  accuracy: {ae['accuracy']:.4f}  F1: {ae['f1']:.4f}")
print(f"  CLS accuracy: {cls['accuracy']:.4f}  F1: {cls['macro_f1']:.4f}  AUC: {cls['roc_auc']:.4f}")

## Render Markdown report

In [ ]:
template_str = """
# Anomaly Detection for Network Security — Evaluation Report

## Abstract

This study implements and compares two anomaly detection approaches on the NSL-KDD benchmark
dataset: an unsupervised autoencoder that flags traffic whose reconstruction error exceeds a
normal-traffic percentile threshold, and a supervised feedforward classifier trained end-to-end
over five traffic categories. Both models are implemented in PyTorch and evaluated on the
standard NSL-KDD test split.

---

## Dataset Description

**Source:** NSL-KDD (improved KDD Cup 1999 benchmark).  
**Training set:** ~125 000 samples · **Test set:** ~22 000 samples  
**Features:** 41 attributes (TCP connection, content, traffic statistics)  
**Classes:** normal, DoS, Probe, R2L, U2R

---

## Methodology

### Preprocessing
- Categorical features (`protocol_type`, `service`, `flag`) encoded with `LabelEncoder` fitted jointly on train + test.
- Numerical features standardised with `StandardScaler` (fit on train only).
- 10 % of training data held out for validation (stratified).

### Autoencoder
- Architecture: 41 → 64 → 48 → **32** (latent) → 48 → 64 → 41, ReLU + BatchNorm.
- Trained on **normal-only** traffic; loss = MSE reconstruction error.
- Decision threshold = 95th percentile of normal training reconstruction errors.

### Classifier
- Architecture: 41 → 128 → 64 → 32 → 5, ReLU + BatchNorm + Dropout(0.3).
- Trained on full labelled dataset; loss = CrossEntropy over 5 classes.

Both models: Adam optimiser, lr=1e-3, batch=256, early stopping (patience=7).

---

## Results

### Training Curves

![Autoencoder Loss](../artifacts/ae_loss_curves.png)
![Classifier Loss](../artifacts/cls_loss_curves.png)
![Classifier Accuracy](../artifacts/cls_accuracy_curves.png)

### Autoencoder — Binary Metrics (normal vs. attack)

| Metric    | Value |
|-----------|-------|
| Accuracy  | {{ '%.4f' % ae.accuracy }} |
| Precision | {{ '%.4f' % ae.precision }} |
| Recall    | {{ '%.4f' % ae.recall }} |
| F1        | {{ '%.4f' % ae.f1 }} |

![Reconstruction Error Distribution](../artifacts/recon_error_dist.png)

### Classifier — Multi-class Metrics

| Metric    | Value |
|-----------|-------|
| Accuracy  | {{ '%.4f' % cls.accuracy }} |
| Macro F1  | {{ '%.4f' % cls.macro_f1 }} |
| ROC-AUC   | {{ '%.4f' % cls.roc_auc }} |

![Confusion Matrix](../artifacts/confusion_matrix.png)
![ROC Curves](../artifacts/roc_curves.png)

### Comparison: Autoencoder vs. Classifier (binary normal/attack)

| Model | Accuracy | Precision | Recall | F1 |
|-------|----------|-----------|--------|----|
| Autoencoder (unsupervised) | {{ '%.4f' % ae.accuracy }} | {{ '%.4f' % ae.precision }} | {{ '%.4f' % ae.recall }} | {{ '%.4f' % ae.f1 }} |
| Classifier (supervised) | {{ '%.4f' % cls.binary.accuracy }} | {{ '%.4f' % cls.binary.precision }} | {{ '%.4f' % cls.binary.recall }} | {{ '%.4f' % cls.binary.f1 }} |

### t-SNE Latent Space

![t-SNE Latent Space](../artifacts/tsne_latent.png)

---

## Discussion

The supervised classifier achieves higher precision and F1 by leveraging ground-truth labels
during training. The autoencoder, trained without any attack labels, still demonstrates
meaningful separation of normal and attack traffic in its latent space (visible in the t-SNE
plot), and achieves competitive recall — critical for intrusion detection where missing attacks
is costly. The threshold selection strategy (95th percentile) provides a tunable knob for the
precision/recall trade-off.

---

## Conclusion

Both paradigms are viable for network intrusion detection. Supervised classifiers outperform
unsupervised autoencoders when labelled data is available, but autoencoders are valuable for
detecting novel attack types not seen during training. Future work could explore semi-supervised
approaches, anomaly ensembles, or online learning for evolving network traffic patterns.
"""

from jinja2 import Template
tmpl = Template(template_str)
report_md = tmpl.render(ae=type('AE', (), ae)(), cls=type('CLS', (), {**cls, 'binary': type('B', (), cls['binary'])()})())

report_path = REPORTS_DIR / 'report.md'
report_path.write_text(report_md)
print(f'Saved: {report_path}')

## Convert to PDF

In [ ]:
pdf_path = REPORTS_DIR / 'report.pdf'
try:
    from md2pdf.core import md2pdf
    md2pdf(str(pdf_path), md_file_path=str(report_path))
    print(f'PDF saved: {pdf_path}')
    assert pdf_path.exists(), 'PDF was not created!'
except Exception as e:
    print(f'PDF generation failed: {e}')
    print('Markdown report is available at:', report_path)